# 문항 1 뒤죽박죽인 날짜 표기를 하나로 통일하기

In [19]:
import re
from datetime import datetime

In [33]:
samples = [
    "2024.12.24",          "2024-12-24",        "2024/12/24",
    "24.12.24",            "2024년 12월 24일",   "2024년 3월 5일",
    "12/24/2024",          "2024.12.24 14:30",  "등록일 : 2024.12.24",
    "2024-13-45",          "작성일 없음",         "",
]

In [35]:
def normalize_date(s) -> str | None:
    # Patterns

    # try america date format first 12/24/2024
    pattern_america = re.compile(r"(?P<Month>\d{1,2})[.\-/](?P<Day>\d{1,2})[.\-/](?P<Year>\d{4})")

    # ymd_dhs 2024.12.24 / 2024-12-24 / 2024/12/24 / 24.12.24 / 2024.12.24 14:30 / 등록일 : 2024.12.24
    pattern_ymd_dhs = re.compile(r"(?P<Year>\d{2}|\d{4})[.\-/](?P<Month>\d{1,2})[.\-/](?P<Day>\d{1,2})")

    # ymd korean 2024년 12월 24일 / 2024년 3월 5일
    pattern_ymd_korean = re.compile(r"(?P<Year>\d{2}|\d{4})년\s*(?P<Month>\d{1,2})월\s*(?P<Day>\d{1,2})일")

    if pattern_america.search(s):
        t = pattern_america.search(s)

        year = t.group("Year")
        month = t.group("Month")
        if len(month) == 1:
            month = "0" + month
        day = t.group("Day")
        if len(day) == 1:
            day = "0" + day

    elif pattern_ymd_dhs.search(s):
        t = pattern_ymd_dhs.search(s)

        year = t.group("Year")
        if len(year) == 2:
            year = "20" + year
        month = t.group("Month")
        if len(month) == 1:
            month = "0" + month
        day = t.group("Day")
        if len(day) == 1:
            day = "0" + day
        
    elif pattern_ymd_korean.search(s):
        t = pattern_ymd_korean.search(s)

        year = t.group("Year")
        if len(year) == 2:
            year = "20" + year
        month = t.group("Month")
        if len(month) == 1:
            month = "0" + month
        day = t.group("Day")
        if len(day) == 1:
            day = "0" + day

    else:
        return None

    date = f"{year}-{month.zfill(2)}-{day.zfill(2)}"

    # try to see if the date is valid
    try:
        datetime.strptime(date, "%Y-%m-%d")
        return date
    except ValueError:
        return None

In [36]:
for sample in samples:
    print(f"{sample} -> {normalize_date(sample)}")

2024.12.24 -> 2024-12-24
2024-12-24 -> 2024-12-24
2024/12/24 -> 2024-12-24
24.12.24 -> 2024-12-24
2024년 12월 24일 -> 2024-12-24
2024년 3월 5일 -> 2024-03-05
12/24/2024 -> 2024-12-24
2024.12.24 14:30 -> 2024-12-24
등록일 : 2024.12.24 -> 2024-12-24
2024-13-45 -> None
작성일 없음 -> None
 -> None


# 문항 2 서버 액세스 로그 파싱과 집계

In [ ]:
import re
import pandas as pd

data = {
    "ip": [],
    "timestamp": [],
    "method": [],
    "path": [],
    "status": [],
    "bytes": [],
    "user_agent": []
}

pattern = re.compile(r'(?P<ip>\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}) - - \[(?P<timestamp>.*?)\] "(?P<method>\w+) (?P<path>.*?) HTTP/1.1" (?P<status>\d{3}) (?P<bytes>\d+) "(.*?)" "(?P<user_agent>.*?)"')

# 추출

parse_count = 0
miss_count = 0
for line in open("access.log"):
    match = pattern.match(line)
    if match:
        parse_count += 1
        for key in data:
            data[key].append(match.group(key))
    else:
        miss_count += 1

# 추출값 저장

df = pd.DataFrame(data)
df.to_csv("access_report.csv", index=False)

# -- 결과물 출력창 --

# 파싱/건너뜀
print(f"파싱 {parse_count}줄 · 건너뜀 {miss_count}줄")

# 상태코드 count
print("\n── 상태코드별 요청 수 ──")
status_count = df["status"].sort_values().value_counts()
print(status_count.to_string())

# 4xx 5xx
print("\n── 4xx·5xx 발생 경로 상위 5 ──")
filtered_df = df[df["status"].astype(str).str.startswith(("4", "5"))]
top_five_paths = filtered_df["path"].value_counts().sort_values(ascending=False).head(5)
print(top_five_paths.to_string())

# 봇 의심
print("\n── 봇 의심 User-Agent ──")
bot_pattern = re.compile(r"bot|crawler|spider|python-requests", re.IGNORECASE)
bot_user_agents = df[df["user_agent"].str.contains(bot_pattern)]
bot_user_agents_count = bot_user_agents["user_agent"].value_counts()
print(bot_user_agents_count.to_string())
print(f"  봇 요청 비율 {bot_user_agents_count.sum() / len(df) * 100:.1f}%")


파싱 3줄 · 건너뜀 0줄

── 상태코드별 요청 수 ──
status
200    1
404    1
429    1

── 4xx·5xx 발생 경로 상위 5 ──
path
/api/search     1
/detail/9981    1

── 봇 의심 User-Agent ──
user_agent
python-requests/2.31.0    1
  봇 요청 비율 33.3%
